In [ ]:
import os
from pathlib import Path

import polars as pl
import polars_st as st

In [ ]:
db_uri = os.environ["DATABASE_URL"]

In [ ]:
# Realtime ingestion config and cache toggle.
#
# Set this to True to reuse parquet snapshots.
# Set this to False to pull fresh data from Postgres and re-save parquet snapshots.
READ_FROM_PARQUET = True
WRITE_PARQUET_ON_DB_READ = True

ROUTES = ["4", "3", "A", "C"]
DIRECTION = 1  # MTA subway: 1 = southbound
WINDOW_DAYS = 14

PARQUET_DIR = Path("data") / "continuous_trajectories"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

RT_TRIP_PARQUET = PARQUET_DIR / "realtime_trip_raw.parquet"
RT_STOP_TIME_PARQUET = PARQUET_DIR / "realtime_stop_time_raw.parquet"

if READ_FROM_PARQUET:
    missing = [p for p in [RT_TRIP_PARQUET, RT_STOP_TIME_PARQUET] if not p.exists()]
    if missing:
        missing_str = "\n".join(f"  - {p}" for p in missing)
        raise FileNotFoundError(
            "READ_FROM_PARQUET is True, but snapshot files are missing:\n"
            f"{missing_str}\n"
            "Set READ_FROM_PARQUET = False to pull from Postgres and create them."
        )

    df_rt_trip_raw = pl.read_parquet(RT_TRIP_PARQUET)
    df_rt_stop_time_raw = pl.read_parquet(RT_STOP_TIME_PARQUET)
    realtime_source = "parquet"
else:
    routes_literal = ",".join(f"'{r}'" for r in ROUTES)

    query_rt_trip = f"""
    SELECT id::text AS trip_id,
           source,
           route_id,
           direction,
           created_at
    FROM realtime.trip
    WHERE source = 'mta_subway'
      AND route_id IN ({routes_literal})
      AND direction = {DIRECTION}
      AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    """

    query_rt_stop_time = f"""
    WITH filtered_trips AS (
        SELECT id
        FROM realtime.trip
        WHERE source = 'mta_subway'
          AND route_id IN ({routes_literal})
          AND direction = {DIRECTION}
          AND created_at >= NOW() - INTERVAL '{WINDOW_DAYS} days'
    )
    SELECT st.trip_id::text AS trip_id,
           st.stop_id,
           st.arrival
    FROM realtime.stop_time st
    JOIN filtered_trips ft ON ft.id = st.trip_id
    ORDER BY st.trip_id, st.arrival
    """

    df_rt_trip_raw = pl.read_database_uri(query_rt_trip, db_uri)
    df_rt_stop_time_raw = pl.read_database_uri(query_rt_stop_time, db_uri)
    realtime_source = "database"

    if WRITE_PARQUET_ON_DB_READ:
        df_rt_trip_raw.write_parquet(RT_TRIP_PARQUET)
        df_rt_stop_time_raw.write_parquet(RT_STOP_TIME_PARQUET)

df_rt = (
    df_rt_stop_time_raw.join(df_rt_trip_raw, on="trip_id", how="inner")
    .select(["trip_id", "route_id", "direction", "created_at", "stop_id", "arrival"])
    .sort(["trip_id", "arrival"])
)

print(f"Realtime source: {realtime_source}")
print(
    f"Loaded {df_rt.height} stop-time rows across "
    f"{df_rt['trip_id'].n_unique()} trips "
    f"on routes {ROUTES} direction={DIRECTION} over the last {WINDOW_DAYS} days"
)
df_rt.head()

In [ ]:
pl.Config.set_tbl_rows(100)
df_rt.filter(trip_id="019d87a5-e1a3-7c63-aaa2-9020f40411f1")

In [ ]:
query = """
SELECT id,
       source,
       long_name,
       short_name,
       color,
       data,
       ST_AsEWKB(geom) AS geometry
FROM static.route
WHERE source = 'mta_subway' AND geom IS NOT NULL
"""
# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "routes.parquet")

df_routes = pl.read_parquet(PARQUET_DIR / "routes.parquet")

In [ ]:
query = """
SELECT id,
       source,
       name,
       ST_AsEWKB(geom) AS geometry,
       data
FROM static.stop
WHERE source = 'mta_subway' AND geom IS NOT NULL
"""

# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "stops.parquet")
df_stops = pl.read_parquet(PARQUET_DIR / "stops.parquet")

In [ ]:
query = """
SELECT route_id,
       stop_id,
       source,
       stop_sequence,
       data
FROM static.route_stop
WHERE source = 'mta_subway'
    AND COALESCE(data->>'stop_type', '') IN ('full_time', 'part_time')
"""

# pl.read_database_uri(query, db_uri).write_parquet(PARQUET_DIR / "route_stops.parquet")
df_route_stops = pl.read_parquet(PARQUET_DIR / "route_stops.parquet")

In [ ]:
# 1. Join stops to route_stops to get the stop geometries in order
df_stops_ordered = df_route_stops.join(
    df_stops.select(["id", "geometry"]), left_on="stop_id", right_on="id", how="left"
).rename({"geometry": "stop_geom"})  # Rename to avoid conflict with route geometry

# Sort by route and stop sequence to ensure they are in the correct travel order
df_stops_ordered = df_stops_ordered.sort(["route_id", "stop_sequence"])

# 2. Join the ordered stops with the route geometries
df_joined = df_stops_ordered.join(
    df_routes.select(["id", "geometry"]), left_on="route_id", right_on="id", how="left"
).rename({"geometry": "route_geom"})

# 3. Project stops onto the route line to get fractional distances
# st.project with normalized=True returns a float between 0.0 and 1.0 representing how far along the line the point is
df_projected = df_joined.with_columns(
    pl.col("route_geom")
    .st.project(pl.col("stop_geom"), normalized=True)
    .alias("fraction_start")
)

# 4. Use Polars window functions to get the destination stop for each segment
df_segments = df_projected.with_columns(
    pl.col("fraction_start").shift(-1).over("route_id").alias("fraction_end"),
    pl.col("stop_id").shift(-1).over("route_id").alias("next_stop_id"),
)

# Drop the last stop of each route, as it doesn't have a "next" stop to form a segment
df_segments = df_segments.drop_nulls(subset=["fraction_end"])

# 5. Extract the geometry between the start and end fractions
df_segments = df_segments.with_columns(
    pl.col("route_geom")
    .st.substring(pl.col("fraction_start"), pl.col("fraction_end"))
    .alias("segment_geom")
)

# df_final_segments = df_segments.select(
#     ["route_id", "stop_sequence", "stop_id", "next_stop_id", "segment_geom"]
# )

In [ ]:
import heapq
from collections import defaultdict
import datetime as dt

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq


def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371008.8  # mean Earth radius in meters
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0) ** 2
    return float(2.0 * r * np.arcsin(np.sqrt(a)))


# Build stop coordinate lookup (WGS84 lon/lat from point geometry).
stop_coords_df = (
    df_stops.select(["id", "geometry"])
    .with_columns(
        lon=pl.col("geometry").st.x(),
        lat=pl.col("geometry").st.y(),
    )
    .select(["id", "lat", "lon"])
    .drop_nulls(["lat", "lon"])
    .unique(subset=["id"], keep="first")
)
stop_coord_map = {
    row["id"]: (float(row["lat"]), float(row["lon"]))
    for row in stop_coords_df.iter_rows(named=True)
}

# Build per-route ordered stop list from static route_stop.
route_stop_rows = (
    df_route_stops.filter(pl.col("route_id").is_in(ROUTES))
    .select(["route_id", "stop_id", "stop_sequence"])
    .sort(["route_id", "stop_sequence"])
)

route_order_rows: dict[str, list[tuple[int, str]]] = defaultdict(list)
for row in route_stop_rows.iter_rows(named=True):
    sid = row["stop_id"]
    if sid not in stop_coord_map:
        continue
    route_order_rows[row["route_id"]].append((int(row["stop_sequence"]), sid))

# Build an undirected weighted graph of adjacent route_stop pairs for each route.
route_graph: dict[str, dict[str, dict[str, float]]] = {}
for route_id, seq_rows in route_order_rows.items():
    adj: dict[str, dict[str, float]] = defaultdict(dict)
    for (_, u), (_, v) in zip(seq_rows, seq_rows[1:]):
        if u == v:
            continue
        lat1, lon1 = stop_coord_map[u]
        lat2, lon2 = stop_coord_map[v]
        w = haversine_m(lat1, lon1, lat2, lon2)
        prev = adj[u].get(v)
        if prev is None or w < prev:
            adj[u][v] = w
            adj[v][u] = w
    route_graph[route_id] = {k: dict(v) for k, v in adj.items()}

_shortest_path_cache: dict[tuple[str, str, str], float | None] = {}


def shortest_path_distance(
    route_id: str, start_stop: str, end_stop: str
) -> float | None:
    """Shortest-path distance in meters on the route stop graph."""
    if start_stop == end_stop:
        return 0.0
    key = (route_id, start_stop, end_stop)
    if key in _shortest_path_cache:
        return _shortest_path_cache[key]

    g = route_graph.get(route_id, {})
    if start_stop not in g or end_stop not in g:
        _shortest_path_cache[key] = None
        return None

    dist: dict[str, float] = {start_stop: 0.0}
    pq: list[tuple[float, str]] = [(0.0, start_stop)]

    while pq:
        d, u = heapq.heappop(pq)
        if u == end_stop:
            _shortest_path_cache[key] = d
            _shortest_path_cache[(route_id, end_stop, start_stop)] = d
            return d
        if d > dist.get(u, float("inf")):
            continue
        for v, w in g.get(u, {}).items():
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))

    _shortest_path_cache[key] = None
    return None


# Build a monotone static stop-distance table for diagnostics and joins.
# Distance is cumulative shortest-path along stop_sequence order.
rows: list[dict] = []
for route_id, seq_rows in route_order_rows.items():
    seen: set[str] = set()
    route_rows: list[dict] = []
    prev_stop: str | None = None
    cum_m = 0.0

    for stop_sequence, stop_id in seq_rows:
        if stop_id in seen:
            continue
        seen.add(stop_id)

        if prev_stop is not None:
            d = shortest_path_distance(route_id, prev_stop, stop_id)
            if d is None:
                lat1, lon1 = stop_coord_map[prev_stop]
                lat2, lon2 = stop_coord_map[stop_id]
                d = haversine_m(lat1, lon1, lat2, lon2)
            cum_m += float(d)

        route_rows.append(
            {
                "route_id": route_id,
                "stop_id": stop_id,
                "stop_sequence": int(stop_sequence),
                "cum_distance_m": float(cum_m),
            }
        )
        prev_stop = stop_id

    route_length_m = float(route_rows[-1]["cum_distance_m"]) if route_rows else 0.0
    for r in route_rows:
        r["route_length_m"] = route_length_m
        r["fraction"] = (
            r["cum_distance_m"] / route_length_m if route_length_m > 0 else 0.0
        )
    rows.extend(route_rows)

df_stops_with_dist = pl.DataFrame(rows).sort(["route_id", "stop_sequence"])

route_graph_summary = []
for route_id in ROUTES:
    g = route_graph.get(route_id, {})
    n_nodes = len(g)
    n_edges = sum(len(vs) for vs in g.values()) // 2
    route_len = (
        float(
            df_stops_with_dist.filter(pl.col("route_id") == route_id)[
                "route_length_m"
            ].max()
        )
        if df_stops_with_dist.filter(pl.col("route_id") == route_id).height > 0
        else 0.0
    )
    route_graph_summary.append(
        {
            "route_id": route_id,
            "graph_nodes": n_nodes,
            "graph_edges": n_edges,
            "graph_route_length_km": route_len / 1000.0,
        }
    )

print(
    f"Graph-based stop distance table: {df_stops_with_dist.height} rows across "
    f"{df_stops_with_dist['route_id'].n_unique()} routes"
)
display(pl.DataFrame(route_graph_summary).sort("route_id"))
df_stops_with_dist.head(10)

In [ ]:
# Geometry sanity: how many distinct projected distances exist per route?
# If a route has many stops but very few distinct cum_distance_m values,
# distance-dedupe will collapse trips to a tiny stop count.

route_geom_quality = (
    df_stops_with_dist.group_by("route_id")
    .agg(
        n_rows=pl.len(),
        n_stops=pl.col("stop_id").n_unique(),
        n_unique_distance=pl.col("cum_distance_m").n_unique(),
        n_unique_distance_round1=pl.col("cum_distance_m").round(1).n_unique(),
        n_unique_distance_round0=pl.col("cum_distance_m").round(0).n_unique(),
    )
    .with_columns(
        uniq_ratio=(pl.col("n_unique_distance") / pl.col("n_stops")),
        uniq_ratio_r1=(pl.col("n_unique_distance_round1") / pl.col("n_stops")),
    )
    .sort("route_id")
)

print("=== Route geometry projection uniqueness ===")
display(route_geom_quality.sort("uniq_ratio"))

# print()
# print("=== Route projected distance frequencies (top 12) ===")
# display(
#     df_stops_with_dist.filter(pl.col("route_id") == "4")
#     .group_by(pl.col("cum_distance_m").round(1).alias("s_round1_m"))
#     .agg(n=pl.len(), sample_stops=pl.col("stop_id").head(5))
#     .sort("n", descending=True)
#     .head(12)
# )


In [ ]:
print(f"Realtime source in use: {realtime_source}")
print(
    f"Trip rows: {df_rt_trip_raw.height:,} | "
    f"Stop-time rows: {df_rt_stop_time_raw.height:,} | "
    f"Joined rows: {df_rt.height:,}"
)
print(
    f"Trips: {df_rt['trip_id'].n_unique():,} | "
    f"Routes: {df_rt['route_id'].n_unique():,} | "
    f"Window: {WINDOW_DAYS} day(s)"
)
df_rt.head()

Attach cumulative distance to each arrival, group into per-trip arrays,
and filter to trips that are usable for cubic-spline fitting.

Requirements for a usable trip:

- at least 4 stops (CubicSpline needs >= 4 knots for a meaningful fit)
- strictly monotone arrival timestamps (time must be a function of stop)
- monotone cumulative distance (train only moves forward along the route).
  The stored route LineString may be oriented either way relative to the
  direction of travel, so a trip with strictly _decreasing_ s is equally
  valid: we simply flip it to strictly increasing via s' = L - s.

Small tolerances handle realtime-feed quirks where two consecutive stops
share the same predicted arrival second, or where st.project rounding
puts two stops at near-identical fractions.


In [ ]:
TIME_TOL_S = 0.5  # seconds; dedupe arrivals within this window
DIST_TOL_M = 1.0  # meters; dedupe stops within this much of each other


def _classify_monotone(xs: list[float], tol: float) -> str:
    """Return 'inc', 'dec', or 'none' after a tol-tolerant dedupe check."""
    diffs = np.diff(xs)
    if np.all(diffs > -tol) and np.any(diffs > tol):
        return "inc"
    if np.all(diffs < tol) and np.any(diffs < -tol):
        return "dec"
    return "none"


# Build per-trip cumulative distance directly from stop-to-stop shortest paths

# on the route stop graph (robust to skipped stops and branch detours).
route_length_lookup = {
    row["route_id"]: float(row["route_length_m"])
    for row in (
        df_stops_with_dist.group_by("route_id")
        .agg(pl.col("route_length_m").max().alias("route_length_m"))
        .iter_rows(named=True)
    )
}

df_trips_input = (
    df_rt.filter(pl.col("route_id").is_in(ROUTES), pl.col("direction") == DIRECTION)
    .select(["trip_id", "route_id", "direction", "created_at", "stop_id", "arrival"])
    .sort(["trip_id", "arrival"])
)

grouped_rt = df_trips_input.group_by(["trip_id", "route_id", "direction"]).agg(
    pl.col("created_at"),
    pl.col("arrival"),
    pl.col("stop_id"),
)

rt_geo_rows: list[dict] = []
trip_graph_gap_rows: list[dict] = []

for row in grouped_rt.iter_rows(named=True):
    trip_id = row["trip_id"]
    route_id = row["route_id"]
    direction = row["direction"]

    created_at = list(row["created_at"])
    arrivals = list(row["arrival"])
    stop_ids = list(row["stop_id"])

    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    created_at = [created_at[i] for i in order]
    arrivals = [arrivals[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    if not arrivals:
        continue

    cum_s: list[float] = [0.0]
    n_pairs = 0
    n_missing_paths = 0

    for i in range(1, len(stop_ids)):
        prev_stop = stop_ids[i - 1]
        curr_stop = stop_ids[i]
        n_pairs += 1

        d = shortest_path_distance(route_id, prev_stop, curr_stop)
        if d is None:
            n_missing_paths += 1
            if prev_stop in stop_coord_map and curr_stop in stop_coord_map:
                lat1, lon1 = stop_coord_map[prev_stop]
                lat2, lon2 = stop_coord_map[curr_stop]
                d = haversine_m(lat1, lon1, lat2, lon2)
            else:
                d = 0.0

        cum_s.append(cum_s[-1] + float(d))

    route_length_m = route_length_lookup.get(route_id, max(cum_s[-1], 1.0))
    if route_length_m <= 0:
        route_length_m = max(cum_s[-1], 1.0)

    trip_graph_gap_rows.append(
        {
            "trip_id": trip_id,
            "route_id": route_id,
            "n_pairs": n_pairs,
            "n_missing_paths": n_missing_paths,
        }
    )

    for i in range(len(arrivals)):
        rt_geo_rows.append(
            {
                "trip_id": trip_id,
                "route_id": route_id,
                "direction": direction,
                "created_at": created_at[i],
                "stop_id": stop_ids[i],
                "arrival": arrivals[i],
                "cum_distance_m": float(cum_s[i]),
                "route_length_m": float(route_length_m),
            }
        )

df_rt_geo = pl.DataFrame(rt_geo_rows).sort(["trip_id", "arrival"])
,
df_trips_grouped = (
    df_rt_geo.group_by(["trip_id", "route_id", "direction"])
    .agg(
        pl.col("arrival"),
        pl.col("cum_distance_m"),
        pl.col("stop_id"),
        pl.col("route_length_m").first().alias("route_length_m"),
    )
    .with_columns(n_stops=pl.col("arrival").list.len())
)

trip_graph_gap_df = pl.DataFrame(trip_graph_gap_rows)
gap_summary = (
    trip_graph_gap_df.select(
        pl.len().alias("n_trips"),
        pl.col("n_missing_paths").sum().alias("total_missing_paths"),
        ((pl.col("n_missing_paths") > 0).sum()).alias("trips_with_any_missing_path"),
    )
    if trip_graph_gap_df.height > 0
    else pl.DataFrame(
        [{"n_trips": 0, "total_missing_paths": 0, "trips_with_any_missing_path": 0}]
    )
)

valid_trips: list[dict] = []
n_total = df_trips_grouped.height
n_too_short = 0
n_nonmono_time = 0
n_nonmono_dist = 0
n_flipped = 0
example_bad: dict | None = None

for row in df_trips_grouped.iter_rows(named=True):
    arrivals = list(row["arrival"])
    s_m = list(row["cum_distance_m"])
    stop_ids = list(row["stop_id"])
    L = float(row["route_length_m"])

    # Sort by arrival (should already be sorted) and dedupe ties within TIME_TOL_S.
    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    arrivals = [arrivals[i] for i in order]
    s_m = [s_m[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    a_sec = [(a - arrivals[0]).total_seconds() for a in arrivals]
    keep_idx = [0]
    for i in range(1, len(a_sec)):
        if a_sec[i] - a_sec[keep_idx[-1]] > TIME_TOL_S:
            keep_idx.append(i)
    arrivals = [arrivals[i] for i in keep_idx]
    s_m = [s_m[i] for i in keep_idx]
    stop_ids = [stop_ids[i] for i in keep_idx]
    n = len(arrivals)

    if n < 4:
        n_too_short += 1
        continue

    # Time must now be strictly increasing after the tol dedupe.
    if not all(arrivals[i] > arrivals[i - 1] for i in range(1, n)):
        n_nonmono_time += 1
        if example_bad is None:
            example_bad = {"reason": "time", "arrivals": arrivals[:6], "s_m": s_m[:6]}
        continue

    # Accept either strictly-increasing or strictly-decreasing distance.
    kind = _classify_monotone(s_m, DIST_TOL_M)
    if kind == "none":
        n_nonmono_dist += 1
        if example_bad is None:
            example_bad = {
                "reason": "dist",
                "stop_ids": stop_ids[:8],
                "s_m": [round(x, 1) for x in s_m[:8]],
            }
        continue
    if kind == "dec":
        s_m = [L - x for x in s_m]
        n_flipped += 1

    # After a potential flip the sequence should be (approximately) increasing;
    # enforce strict monotonicity by dropping any tol-ties that remain.
    keep = [0]
    for i in range(1, len(s_m)):
        if s_m[i] - s_m[keep[-1]] > DIST_TOL_M:
            keep.append(i)
    arrivals = [arrivals[i] for i in keep]
    s_m = [s_m[i] for i in keep]
    stop_ids = [stop_ids[i] for i in keep]
    n = len(arrivals)
    if n < 4:
        n_too_short += 1
        continue

    valid_trips.append(
        {
            "trip_id": row["trip_id"],
            "route_id": row["route_id"],
            "direction": row["direction"],
            "arrivals": arrivals,
            "s_m": np.asarray(s_m, dtype=float),
            "stop_ids": stop_ids,
            "route_length_m": L,
            "n_stops": n,
            "reversed": kind == "dec",
        }
    )

print("=== Graph distance construction ===")
print(
    f"Trips built: {int(gap_summary['n_trips'][0]):,} | "
    f"Trips with missing path links: {int(gap_summary['trips_with_any_missing_path'][0]):,} | "
    f"Total missing links: {int(gap_summary['total_missing_paths'][0]):,}"
)

print()
print(f"Total trips considered:              {n_total}")
print(f"Dropped - fewer than 4 stops:         {n_too_short}")
print(f"Dropped - non-monotone arrival times: {n_nonmono_time}")
print(f"Dropped - non-monotone distances:     {n_nonmono_dist}")
print(f"Flipped (decreasing -> increasing):   {n_flipped}")
print(f"Valid trips kept:                     {len(valid_trips)}")
if example_bad is not None and len(valid_trips) == 0:
    print()
    print("Example trip that was dropped:")
    print(example_bad)
if valid_trips:
    _n_stops = [t["n_stops"] for t in valid_trips]
    print(
        f"Stops per trip: min={min(_n_stops)} median={int(np.median(_n_stops))} "
        f"max={max(_n_stops)}"
    )

In [ ]:
# Diagnostic: where do stops get lost?
# This does not mutate valid_trips; it only prints data-quality summaries.

# 1) Raw trip/update shape before geo join logic
raw_trip_stats = (
    df_rt.group_by(["trip_id", "route_id", "direction"])
    .agg(
        n_rows=pl.len(),
        n_unique_stops=pl.col("stop_id").n_unique(),
        first_arrival=pl.col("arrival").min(),
        last_arrival=pl.col("arrival").max(),
    )
    .with_columns(
        span_minutes=(
            (pl.col("last_arrival") - pl.col("first_arrival")).dt.total_seconds() / 60.0
        )
    )
)

# 2) Check if route_id+stop_id mapping is one-to-one or fanout (join explosion risk)
route_stop_multiplicity = (
    df_stops_with_dist.group_by(["route_id", "stop_id"])
    .len()
    .rename({"len": "matches"})
)
rt_with_mult = df_rt.join(
    route_stop_multiplicity, on=["route_id", "stop_id"], how="left"
)

join_fanout_rows = int(rt_with_mult.filter(pl.col("matches") > 1).height)
join_total_rows = int(rt_with_mult.height)

# 3) Reproduce filtering stages and keep per-trip counts at each stage
rows = []
for row in df_trips_grouped.iter_rows(named=True):
    arrivals = list(row["arrival"])
    s_m = list(row["cum_distance_m"])
    stop_ids = list(row["stop_id"])
    L = float(row["route_length_m"])

    n0_raw = len(arrivals)

    order = sorted(range(len(arrivals)), key=lambda i: arrivals[i])
    arrivals = [arrivals[i] for i in order]
    s_m = [s_m[i] for i in order]
    stop_ids = [stop_ids[i] for i in order]

    a_sec = [(a - arrivals[0]).total_seconds() for a in arrivals] if arrivals else []
    keep_idx = [0] if arrivals else []
    for i in range(1, len(a_sec)):
        if a_sec[i] - a_sec[keep_idx[-1]] > TIME_TOL_S:
            keep_idx.append(i)

    arrivals = [arrivals[i] for i in keep_idx]
    s_m = [s_m[i] for i in keep_idx]
    stop_ids = [stop_ids[i] for i in keep_idx]
    n1_after_time = len(arrivals)

    status = "ok"
    flipped = False

    if n1_after_time < 4:
        status = "too_short_after_time"
    elif not all(arrivals[i] > arrivals[i - 1] for i in range(1, n1_after_time)):
        status = "nonmono_time"
    else:
        kind = _classify_monotone(s_m, DIST_TOL_M)
        if kind == "none":
            status = "nonmono_dist"
        else:
            if kind == "dec":
                s_m = [L - x for x in s_m]
                flipped = True

            keep = [0]
            for i in range(1, len(s_m)):
                if s_m[i] - s_m[keep[-1]] > DIST_TOL_M:
                    keep.append(i)

            arrivals = [arrivals[i] for i in keep]
            s_m = [s_m[i] for i in keep]
            stop_ids = [stop_ids[i] for i in keep]

            if len(arrivals) < 4:
                status = "too_short_after_dist"

    n2_after_dist = len(arrivals)

    rows.append(
        {
            "trip_id": row["trip_id"],
            "route_id": row["route_id"],
            "n0_raw": n0_raw,
            "n1_after_time": n1_after_time,
            "n2_after_dist": n2_after_dist,
            "status": status,
            "flipped": flipped,
        }
    )

diag = pl.DataFrame(rows)
valid_diag = diag.filter(pl.col("status") == "ok")


def q_summary(df: pl.DataFrame, col: str) -> str:
    if df.height == 0:
        return "n=0"
    vals = df[col]
    return (
        f"n={df.height:,} min={int(vals.min())} p25={int(vals.quantile(0.25))} "
        f"median={int(vals.median())} p75={int(vals.quantile(0.75))} max={int(vals.max())}"
    )


print("=== Raw snapshot-level trip/update stats ===")
print(f"Trips (grouped updates): {raw_trip_stats.height:,}")
print(f"n_rows per trip:         {q_summary(raw_trip_stats, 'n_rows')}")
print(f"n_unique_stops per trip: {q_summary(raw_trip_stats, 'n_unique_stops')}")
print(
    f"arrival span (min):      n={raw_trip_stats.height:,} "
    f"median={raw_trip_stats['span_minutes'].median():.2f} "
    f"p75={raw_trip_stats['span_minutes'].quantile(0.75):.2f} "
    f"max={raw_trip_stats['span_minutes'].max():.2f}"
)

print()
print("=== Join multiplicity check (route_id + stop_id) ===")
print(
    f"Rows with >1 geometry match: {join_fanout_rows:,} / {join_total_rows:,} "
    f"({(100.0 * join_fanout_rows / join_total_rows if join_total_rows else 0.0):.2f}%)"
)

print()
print("=== Stop-count attrition through filtering ===")
print(f"Raw n_stops:             {q_summary(diag, 'n0_raw')}")
print(f"After time dedupe:       {q_summary(diag, 'n1_after_time')}")
print(f"After distance dedupe:   {q_summary(diag, 'n2_after_dist')}")
print(f"Valid only (status=ok):  {q_summary(valid_diag, 'n2_after_dist')}")

print()
print("=== Why trips are dropped ===")
display(diag.group_by("status").len().sort("len", descending=True))

print()
print("=== Per-route valid-trip median stops ===")
if valid_diag.height > 0:
    display(
        valid_diag.group_by("route_id")
        .agg(
            n_valid=pl.len(),
            median_stops=pl.col("n2_after_dist").median(),
            p75_stops=pl.col("n2_after_dist").quantile(0.75),
            max_stops=pl.col("n2_after_dist").max(),
        )
        .sort("n_valid", descending=True)
    )
else:
    print("No valid trips.")

Per-trip fitters: Cubic Spline (our target method) and linear baseline.

#

Boundary conditions:

- Natural spline -> s''(t) = 0 at endpoints. Mathematically clean but
  implies the train is still accelerating/decelerating at the terminal.
- Clamped spline -> s'(t) = 0 at endpoints (train stopped at origin/terminal).
  This matches the physics of a subway at the end of a route and tends to
  produce smoother, more realistic v(t) near the endpoints.
  We keep both so we can compare.


In [ ]:
def trip_time_seconds(arrivals: list[dt.datetime]) -> np.ndarray:
    """Convert list of timestamps to seconds since the first arrival."""
    t0 = arrivals[0]
    return np.asarray([(a - t0).total_seconds() for a in arrivals], dtype=float)


def fit_cubic(
    t_seconds: np.ndarray, s_meters: np.ndarray, clamped: bool = True
) -> CubicSpline:
    bc = ((1, 0.0), (1, 0.0)) if clamped else "natural"
    return CubicSpline(t_seconds, s_meters, bc_type=bc, extrapolate=False)


def fit_linear(t_seconds: np.ndarray, s_meters: np.ndarray):
    """Piecewise-linear interpolant; returns a callable mirroring CubicSpline."""
    ts, ss = t_seconds, s_meters

    def f(t):
        return np.interp(t, ts, ss)

    return f


# Demo: fit the longest trip we have and print a quick summary.
demo_trip = max(valid_trips, key=lambda x: x["n_stops"])
demo_t = trip_time_seconds(demo_trip["arrivals"])
demo_s = demo_trip["s_m"]

demo_spline_natural = fit_cubic(demo_t, demo_s, clamped=False)
demo_spline_clamped = fit_cubic(demo_t, demo_s, clamped=True)
demo_linear = fit_linear(demo_t, demo_s)

print(f"Demo trip_id={demo_trip['trip_id']} route={demo_trip['route_id']}")
print(
    f"  {demo_trip['n_stops']} stops, duration={demo_t[-1] / 60:.1f} min, "
    f"distance covered={demo_s[-1] / 1000:.2f} km "
    f"(route length={demo_trip['route_length_m'] / 1000:.2f} km)"
)
print(
    f"  Avg speed = {demo_s[-1] / demo_t[-1]:.2f} m/s "
    f"({demo_s[-1] / demo_t[-1] * 2.237:.1f} mph)"
)

In [ ]:
# Numerical differentiation: CubicSpline.derivative(k) returns the exact
# piecewise-polynomial derivative of order k. For a cubic spline this is an
# analytic derivative of the interpolant (not a finite-difference scheme), so
# there is no truncation error from the differentiation itself; all error is
# inherited from the fit. This is what the proposal calls "numerical
# differentiation applied to the interpolated position function."
#
# Physical plausibility thresholds for NYC subway:
#   - |v| should not exceed ~35 m/s (~78 mph). R142 top speed is ~55 mph.
#   - |a| should not exceed ~1.5 m/s^2 (service acceleration/brake is ~1.0 m/s^2).
# Trips that violate these are usually the result of a missed station
# (inflated apparent speed) or clock skew in the realtime feed.

SANITY_V_MAX = 35.0  # m/s
SANITY_A_MAX = 1.5  # m/s^2

flagged_trips: list[str] = []
v_maxes = []
a_maxes = []

for trip in valid_trips:
    t_sec = trip_time_seconds(trip["arrivals"])
    s_m = trip["s_m"]
    spline = fit_cubic(t_sec, s_m, clamped=True)
    v_fn = spline.derivative(1)
    a_fn = spline.derivative(2)

    # Sample at 500 points across the trip
    ts = np.linspace(t_sec[0], t_sec[-1], 500)
    v = v_fn(ts)
    a = a_fn(ts)

    vmax = float(np.nanmax(np.abs(v)))
    amax = float(np.nanmax(np.abs(a)))
    trip["v_max"] = vmax
    trip["a_max"] = amax
    v_maxes.append(vmax)
    a_maxes.append(amax)

    if vmax > SANITY_V_MAX or amax > SANITY_A_MAX:
        flagged_trips.append(trip["trip_id"])

print(
    f"Physical-plausibility check across {len(valid_trips)} trips "
    f"(|v|>{SANITY_V_MAX} m/s or |a|>{SANITY_A_MAX} m/s^2):"
)
print(
    f"  velocity: median peak = {np.median(v_maxes):.2f} m/s, "
    f"95th = {np.percentile(v_maxes, 95):.2f} m/s, "
    f"max = {np.max(v_maxes):.2f} m/s"
)
print(
    f"  accel:    median peak = {np.median(a_maxes):.3f} m/s^2, "
    f"95th = {np.percentile(a_maxes, 95):.3f} m/s^2, "
    f"max = {np.max(a_maxes):.3f} m/s^2"
)
print(f"  Flagged as implausible: {len(flagged_trips)} / {len(valid_trips)} trips")

In [ ]:
# Leave-one-out cross-validation, the core evaluation from the proposal.
#
# For each trip with at least 5 stops, hide each interior stop i (1..n-2) and
# refit the interpolant on the remaining n-1 points. We then predict the
# arrival time at the hidden station by root-finding s_hat(t) = s_i on the
# bracket [t_{i-1}, t_{i+1}] using scipy.optimize.brentq, and compare against
# the actual arrival t_i.
#
# We run this twice: once for the Cubic Spline (clamped) and once for the
# linear-interp baseline, giving us a direct apples-to-apples comparison.


def cv_errors(trips: list[dict], method: str = "cubic"):
    """Return (errors, per_stop) where per_stop is (relative_position, err_s)."""
    errors: list[float] = []
    per_stop: list[tuple[float, float]] = []
    failures = 0

    for trip in trips:
        t_sec = trip_time_seconds(trip["arrivals"])
        s_m = trip["s_m"]
        n = len(t_sec)
        if n < 5:
            continue

        for i in range(1, n - 1):
            mask = np.ones(n, dtype=bool)
            mask[i] = False
            t_train = t_sec[mask]
            s_train = s_m[mask]

            try:
                if method == "cubic":
                    f = CubicSpline(
                        t_train,
                        s_train,
                        bc_type=((1, 0.0), (1, 0.0)),
                        extrapolate=False,
                    )
                else:

                    def f(t, tt=t_train, ss=s_train):
                        return np.interp(t, tt, ss)

                s_target = float(s_m[i])

                def g(t):
                    return float(f(t)) - s_target

                t_lo, t_hi = float(t_sec[i - 1]), float(t_sec[i + 1])
                g_lo, g_hi = g(t_lo), g(t_hi)

                # brentq needs a sign change across the bracket. If the
                # interpolant overshoots (cubic can) we may not have one;
                # skip those cases and count them as failures.
                if np.isnan(g_lo) or np.isnan(g_hi) or g_lo * g_hi > 0:
                    failures += 1
                    continue

                t_pred = brentq(g, t_lo, t_hi)
                err = t_pred - float(t_sec[i])
                errors.append(err)
                per_stop.append(((i) / (n - 1), err))
            except Exception:
                failures += 1
                continue

    return np.asarray(errors), per_stop, failures


err_cubic, per_cubic, fail_cubic = cv_errors(valid_trips, method="cubic")
err_linear, per_linear, fail_linear = cv_errors(valid_trips, method="linear")

print(
    f"Cross-validation ran on {sum(1 for t in valid_trips if t['n_stops'] >= 5)} "
    f"trips with >=5 stops"
)
print(f"  Cubic:  {len(err_cubic)} predictions, {fail_cubic} skipped (no bracket)")
print(f"  Linear: {len(err_linear)} predictions, {fail_linear} skipped (no bracket)")

In [ ]:
# Summary metrics + side-by-side error distributions.


def describe_errors(name: str, errs: np.ndarray) -> dict:
    if len(errs) == 0:
        print(f"{name}: no predictions")
        return {}
    stats = {
        "N": len(errs),
        "mean": float(np.mean(errs)),
        "median": float(np.median(errs)),
        "MAE": float(np.mean(np.abs(errs))),
        "RMSE": float(np.sqrt(np.mean(errs**2))),
        "P95_abs": float(np.percentile(np.abs(errs), 95)),
        "max_abs": float(np.max(np.abs(errs))),
    }
    print(
        f"{name:8s} N={stats['N']:5d} "
        f"MAE={stats['MAE']:6.2f}s "
        f"RMSE={stats['RMSE']:6.2f}s "
        f"median_bias={stats['median']:+6.2f}s "
        f"P95|err|={stats['P95_abs']:6.2f}s "
        f"max|err|={stats['max_abs']:7.2f}s"
    )
    return stats


print("Leave-one-out arrival-time prediction error")
print("-" * 80)
stats_cubic = describe_errors("Cubic", err_cubic)
stats_linear = describe_errors("Linear", err_linear)

if stats_cubic and stats_linear:
    print()
    print(
        f"Cubic vs linear: MAE improved by "
        f"{(stats_linear['MAE'] - stats_cubic['MAE']):+.2f}s "
        f"({(stats_linear['MAE'] - stats_cubic['MAE']) / stats_linear['MAE'] * 100:+.1f}%), "
        f"RMSE by {(stats_linear['RMSE'] - stats_cubic['RMSE']):+.2f}s "
        f"({(stats_linear['RMSE'] - stats_cubic['RMSE']) / stats_linear['RMSE'] * 100:+.1f}%)"
    )

# Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

clip = 60.0  # seconds; zoom into the bulk of the distribution
bins = np.linspace(-clip, clip, 61)
axes[0].hist(
    np.clip(err_linear, -clip, clip),
    bins=bins,
    alpha=0.55,
    label="Linear",
    color="tab:orange",
)
axes[0].hist(
    np.clip(err_cubic, -clip, clip),
    bins=bins,
    alpha=0.7,
    label="Cubic",
    color="tab:blue",
)
axes[0].axvline(0, color="k", lw=0.5)
axes[0].set_xlabel("Predicted arrival error (s)   [clipped to +/-60]")
axes[0].set_ylabel("Count")
axes[0].set_title("Leave-one-out arrival-time error")
axes[0].legend()

if per_cubic and per_linear:
    pc_rel = np.asarray([p[0] for p in per_cubic])
    pc_abs = np.abs([p[1] for p in per_cubic])
    pl_rel = np.asarray([p[0] for p in per_linear])
    pl_abs = np.abs([p[1] for p in per_linear])
    axes[1].scatter(pl_rel, pl_abs, s=5, alpha=0.25, label="Linear", color="tab:orange")
    axes[1].scatter(pc_rel, pc_abs, s=5, alpha=0.25, label="Cubic", color="tab:blue")
    axes[1].set_xlabel("Relative position along trip (0 = first stop, 1 = last)")
    axes[1].set_ylabel("|Arrival error| (s)")
    axes[1].set_yscale("symlog", linthresh=1.0)
    axes[1].set_title("Error vs hidden-stop position")
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Stability / round-off / truncation analysis.
#
# 1. Input noise: add Gaussian noise with sigma=5s to arrival timestamps and
#    re-fit. Cubic spline interpolation is known to be well-conditioned for
#    reasonably spaced data, but this gives us a concrete sense of how much
#    small timing errors in the realtime feed propagate to position estimates.
#
# 2. Monotonicity: a cubic spline through monotone data is *not* guaranteed
#    to be monotone (overshoot). We flag trips whose fitted s(t) is
#    non-monotone - these are cases where brentq-based arrival prediction
#    can fail and where linear interpolation would actually be safer.
#
# 3. Node spacing / conditioning: the tridiagonal system solved by
#    CubicSpline has condition number that grows with the ratio of the
#    largest to smallest interval (max h_i / min h_i). We report that ratio
#    as a cheap proxy for numerical conditioning.

rng = np.random.default_rng(42)
NOISE_SIGMA = 5.0  # seconds of added Gaussian noise to each arrival

stab_shifts_rel = []  # max |s_noisy - s_clean| / route_length, per trip
nonmono_trips = 0
interval_ratios = []

for trip in valid_trips:
    t_sec = trip_time_seconds(trip["arrivals"])
    s_m = trip["s_m"]
    L = trip["route_length_m"]

    # Conditioning proxy
    h = np.diff(t_sec)
    if h.min() > 0:
        interval_ratios.append(float(h.max() / h.min()))

    spline_clean = fit_cubic(t_sec, s_m, clamped=True)

    # Perturb and re-fit, preserving strict monotonicity of t
    t_noisy = t_sec + rng.normal(0.0, NOISE_SIGMA, size=t_sec.shape)
    for k in range(1, len(t_noisy)):
        if t_noisy[k] <= t_noisy[k - 1]:
            t_noisy[k] = t_noisy[k - 1] + 1e-3
    spline_noisy = fit_cubic(t_noisy, s_m, clamped=True)

    ts = np.linspace(t_sec[0], t_sec[-1], 500)
    s_clean = spline_clean(ts)
    s_noisy = spline_noisy(ts)
    stab_shifts_rel.append(float(np.nanmax(np.abs(s_clean - s_noisy)) / L))

    # Monotonicity of the clean fit
    s_dense = spline_clean(ts)
    if np.any(np.diff(s_dense) < 0):
        nonmono_trips += 1

stab = np.asarray(stab_shifts_rel)
print(f"Input-noise stability (sigma = {NOISE_SIGMA} s per arrival):")
print(
    f"  max |delta s| / route length:  median = {np.median(stab) * 100:.3f}%, "
    f"95th = {np.percentile(stab, 95) * 100:.3f}%, "
    f"max = {np.max(stab) * 100:.3f}%"
)
print()
print(
    f"Monotonicity: {nonmono_trips} / {len(valid_trips)} "
    f"({nonmono_trips / len(valid_trips) * 100:.1f}%) cubic fits have non-monotone s(t)"
)
print(
    f"  These trips exhibit overshoot; arrival-time prediction via brentq can"
    f" fail on them, and linear interpolation is preferable as a fallback."
)
print()
if interval_ratios:
    ir = np.asarray(interval_ratios)
    print(f"Node-spacing conditioning (max h_i / min h_i):")
    print(
        f"  median = {np.median(ir):.2f}, "
        f"95th = {np.percentile(ir, 95):.2f}, "
        f"max = {np.max(ir):.2f}"
    )
    print(
        f"  Large ratios indicate irregular spacing; the tridiagonal solve"
        f" stays well-conditioned as long as this ratio is modest (< ~50)."
    )

In [ ]:
# Visualization 1: position, velocity, and acceleration over time for a
# representative trip. Linear interpolation only gives a position curve;
# its "velocity" is a step function (derivative of linear segments) and its
# "acceleration" is zero everywhere except at knots, which is exactly why
# we need the cubic spline for physically meaningful v and a.

trip = demo_trip  # longest trip, selected earlier
t_sec = trip_time_seconds(trip["arrivals"])
s_m = trip["s_m"]

spline = fit_cubic(t_sec, s_m, clamped=True)
spline_nat = fit_cubic(t_sec, s_m, clamped=False)
v_fn = spline.derivative(1)
a_fn = spline.derivative(2)

ts = np.linspace(t_sec[0], t_sec[-1], 2000)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(ts / 60, spline(ts) / 1000, label="Cubic (clamped)", color="tab:blue")
axes[0].plot(
    ts / 60, spline_nat(ts) / 1000, ":", label="Cubic (natural)", color="tab:cyan"
)
axes[0].plot(
    ts / 60,
    np.interp(ts, t_sec, s_m) / 1000,
    "--",
    label="Linear baseline",
    color="tab:orange",
)
axes[0].scatter(
    t_sec / 60,
    s_m / 1000,
    color="red",
    zorder=5,
    s=30,
    label="Observed arrivals",
)
axes[0].set_ylabel("Distance along route (km)")
axes[0].set_title(
    f"Trip {trip['trip_id'][:8]}..  route {trip['route_id']}  "
    f"({trip['n_stops']} stops, {t_sec[-1] / 60:.1f} min)"
)
axes[0].legend(loc="lower right")
axes[0].grid(alpha=0.3)

axes[1].plot(ts / 60, v_fn(ts), color="tab:blue")
axes[1].axhline(0, color="k", lw=0.5)
axes[1].axhline(
    SANITY_V_MAX, color="red", lw=0.5, ls=":", label=f"|v| = {SANITY_V_MAX}"
)
axes[1].axhline(-SANITY_V_MAX, color="red", lw=0.5, ls=":")
axes[1].set_ylabel("Velocity s'(t)  (m/s)")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)

axes[2].plot(ts / 60, a_fn(ts), color="tab:blue")
axes[2].axhline(0, color="k", lw=0.5)
axes[2].axhline(
    SANITY_A_MAX, color="red", lw=0.5, ls=":", label=f"|a| = {SANITY_A_MAX}"
)
axes[2].axhline(-SANITY_A_MAX, color="red", lw=0.5, ls=":")
axes[2].set_ylabel("Acceleration s''(t)  (m/s^2)")
axes[2].set_xlabel("Time since first stop (min)")
axes[2].legend(loc="upper right")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: map the interpolated train position for the demo trip.
#
# Pipeline:
#   1. At each elapsed time t, evaluate the cubic spline to get s(t) in meters.
#   2. Normalize by the route length to get a fraction f in [0, 1]. If cell 9
#      flipped this trip (stored LineString runs opposite to travel), un-flip
#      to 1 - f so we index the stored line correctly.
#   3. Use polars-st's st.interpolate(f, normalized=True) to resolve the
#      lon/lat point at fractional distance f along the route LineString.
# This is the continuous-position function promised by the proposal.

route_wkb = df_routes.filter(pl.col("id") == trip["route_id"])["geometry"].to_list()[0]
reversed_trip = bool(trip.get("reversed", False))

# Sample N_FRAMES evenly-spaced times and turn each into a fraction along the
# stored route line.
N_FRAMES = 30
sample_times = np.linspace(t_sec[0], t_sec[-1], N_FRAMES)
sample_s = spline(sample_times)
sample_frac = np.clip(sample_s / trip["route_length_m"], 0.0, 1.0)
if reversed_trip:
    sample_frac = 1.0 - sample_frac

points_df = (
    pl.DataFrame(
        {
            "t_sec": sample_times,
            "fraction": sample_frac,
            "route_wkb": [route_wkb] * len(sample_times),
        }
    )
    # TODO: why is this getting converted to wkb in the first place?
    .with_columns(route_geom=st.from_wkb("route_wkb"))
    .with_columns(
        geom=pl.col("route_geom").st.interpolate(pl.col("fraction"), normalized=True),
        label=pl.format("train_t={}min", (pl.col("t_sec") / 60).round(1)),
    )
    .select(["label", "geom"])
)

route_df = (
    pl.DataFrame({"route_wkb": [route_wkb]})
    .with_columns(
        label=pl.lit("route"),
        geom=st.from_wkb("route_wkb"),
    )
    .select(["label", "geom"])
)

viz_df = pl.concat([route_df, points_df], how="vertical")

# "Current" snapshot at 40% through the trip for the printed summary.
elapsed_now = float((t_sec[-1] - t_sec[0]) * 0.4)
s_now = float(spline(elapsed_now))
frac_now = float(np.clip(s_now / trip["route_length_m"], 0.0, 1.0))
frac_line_now = 1.0 - frac_now if reversed_trip else frac_now

now_row = (
    pl.DataFrame({"route_wkb": [route_wkb]})
    .with_columns(route_geom=st.from_wkb("route_wkb"))
    .with_columns(
        pt=pl.col("route_geom").st.interpolate(frac_line_now, normalized=True)
    )
    .with_columns(lon=pl.col("pt").st.x(), lat=pl.col("pt").st.y())
    .row(0, named=True)
)

print(f"At t={elapsed_now:.0f}s ({elapsed_now / 60:.1f} min) after the first stop:")
print(f"  s(t) = {s_now / 1000:.3f} km  ({frac_now * 100:.1f}% of trip distance)")
print(f"  Interpolated position: lon={now_row['lon']:.6f}, lat={now_row['lat']:.6f}")
print(f"  Velocity  s'(t) = {float(v_fn(elapsed_now)):.2f} m/s")
print(f"  Accel     s''(t) = {float(a_fn(elapsed_now)):.3f} m/s^2")

viz_df.st.explore(
    "geom", scatterplot_kwargs={"get_radius": 15, "get_fill_color": "red"}
)

In [ ]:
# Visualization 2.5: Route Geometry, Graph Edges, and Stop Projections
import polars as pl
import polars_st as st
from shapely.geometry import LineString

# 1. Underlay for Routes
routes_wkb = (
    df_routes.filter(pl.col("id").is_in(ROUTES))
    .rename({"geometry": "wkb", "id": "route_id"})
    .with_columns(geom=st.from_wkb("wkb"))
    .select(["route_id", "geom"])
)
m = routes_wkb.st.explore(
    "geom", path_kwargs={"get_color": [160, 160, 160, 180], "width_min_pixels": 3}
)

# 2. Original & Projected Stops
proj_rows = []
for row in df_stops_with_dist.iter_rows(named=True):
    route_id = row["route_id"]
    stop_id = row["stop_id"]
    fraction = row["fraction"]
    lat, lon = stop_coord_map[stop_id]
    proj_rows.append(
        {
            "route_id": route_id,
            "stop_id": stop_id,
            "fraction": fraction,
            "orig_lon": lon,
            "orig_lat": lat,
        }
    )

df_proj = pl.DataFrame(proj_rows).join(routes_wkb, on="route_id")
df_proj = df_proj.with_columns(
    proj_pt=pl.col("geom").st.interpolate(pl.col("fraction"), normalized=True),
    orig_coords=pl.concat_list([pl.col("orig_lon"), pl.col("orig_lat")]),
).with_columns(orig_pt=st.point("orig_coords"))

m.add_layer(
    df_proj.st.explore(
        "orig_pt",
        scatterplot_kwargs={"get_fill_color": [255, 0, 0, 200], "radius_min_pixels": 4},
    )
)
m.add_layer(
    df_proj.st.explore(
        "proj_pt",
        scatterplot_kwargs={"get_fill_color": [0, 255, 0, 200], "radius_min_pixels": 4},
    )
)

# 3. Projection Lines (Original to Projected)
df_proj = df_proj.with_columns(
    proj_lon=pl.col("proj_pt").st.x(), proj_lat=pl.col("proj_pt").st.y()
)
pdf_proj = df_proj.to_pandas()
lines_snap = [
    LineString([(row.orig_lon, row.orig_lat), (row.proj_lon, row.proj_lat)]).wkb
    for _, row in pdf_proj.iterrows()
]

df_snap = pl.DataFrame({"geom": lines_snap}).with_columns(geom=st.from_wkb("geom"))
m.add_layer(
    df_snap.st.explore(
        "geom", path_kwargs={"get_color": [255, 255, 0, 150], "width_min_pixels": 2}
    )
)

# 4. Graph Edges
edges_rows = []
for route_id, g in route_graph.items():
    if route_id not in ROUTES:
        continue
    for u, neighbors in g.items():
        for v, _ in neighbors.items():
            lat1, lon1 = stop_coord_map[u]
            lat2, lon2 = stop_coord_map[v]
            edges_rows.append(LineString([(lon1, lat1), (lon2, lat2)]).wkb)

df_edges = pl.DataFrame({"geom": edges_rows}).with_columns(geom=st.from_wkb("geom"))
m.add_layer(
    df_edges.st.explore(
        "geom", path_kwargs={"get_color": [0, 0, 255, 120], "width_min_pixels": 1}
    )
)

print(
    "Red: Original Stops | Green: Projected Stops | Yellow: Snap Lines | Blue: Graph Edges | Gray: Route Geom"
)
m

In [ ]:
# Visualization 3: animated TripsLayer.
#
# IMPORTANT: TripsLayer only draws data at the layer's `current_time`. You
# must click the Play button on the widget underneath the map to see motion;
# otherwise you're just looking at whichever trains happen to be alive at a
# single frozen instant.
#
# Synchronization: in real wall-clock time each trip has its own start, so at
# any fixed current_time only one or two trains are on screen. To make the
# animation visually informative we time-align every trip to a common epoch
# (SYNC_TRIPS = True). Flip it to False to play in true wall-clock time.
#
# Note: this requires `movingpandas` (added to science/pyproject.toml).
# Run `uv sync` (or `pip install movingpandas`) if the import below fails.

from datetime import datetime, timedelta, timezone

import pandas as pd
import geopandas as gpd
import movingpandas as mpd
import ipywidgets as widgets
from lonboard import Map, TripsLayer, PathLayer

TRIPS_TO_ANIMATE = min(15, len(valid_trips))
N_SAMPLES = 150  # path waypoints per trip
SYNC_TRIPS = True
EPOCH = datetime(2025, 1, 1, tzinfo=timezone.utc)

anim_trips = valid_trips[:TRIPS_TO_ANIMATE]

# Build a long (trip_id, route_id, t, fraction) frame by sampling each trip's
# spline uniformly in time. We un-flip the fraction for reversed trips so it
# indexes the *stored* route line correctly before passing to st.interpolate.
sample_rows: list[dict] = []
for trip in anim_trips:
    t_sec_trip = trip_time_seconds(trip["arrivals"])
    s_m_trip = trip["s_m"]
    spline_trip = fit_cubic(t_sec_trip, s_m_trip, clamped=True)
    times_s = np.linspace(t_sec_trip[0], t_sec_trip[-1], N_SAMPLES)
    s_vals = spline_trip(times_s)
    fracs = np.clip(s_vals / trip["route_length_m"], 0.0, 1.0)
    if trip.get("reversed"):
        fracs = 1.0 - fracs
    t0 = EPOCH if SYNC_TRIPS else trip["arrivals"][0]
    for ts_s, f in zip(times_s, fracs):
        sample_rows.append(
            {
                "trip_id": trip["trip_id"],
                "route_id": trip["route_id"],
                "fraction": float(f),
                "t": t0 + dt.timedelta(seconds=float(ts_s)),
            }
        )

long_df = pl.DataFrame(sample_rows)

# Attach each sample's route geometry and resolve lon/lat via polars-st.
routes_min = df_routes.select(["id", "geometry"]).rename(
    {"id": "route_id", "geometry": "route_wkb"}
)
points_long = (
    long_df.join(routes_min, on="route_id", how="left")
    .with_columns(route_geom=st.from_wkb("route_wkb"))
    .with_columns(
        pt=pl.col("route_geom").st.interpolate(pl.col("fraction"), normalized=True),
    )
    .with_columns(
        lon=pl.col("pt").st.x(),
        lat=pl.col("pt").st.y(),
    )
    .select(["trip_id", "route_id", "t", "lon", "lat"])
    .sort(["trip_id", "t"])
)

# movingpandas wants a GeoDataFrame indexed by timestamp with one row per
# (trajectory, sample) and a traj-id column.
pdf = points_long.to_pandas()
pdf["t"] = pd.to_datetime(pdf["t"], utc=True)
gdf = gpd.GeoDataFrame(
    pdf,
    geometry=gpd.points_from_xy(pdf["lon"], pdf["lat"]),
    crs="EPSG:4326",
).set_index("t")

traj_collection = mpd.TrajectoryCollection(gdf, traj_id_col="trip_id")

t_min_idx, t_max_idx = pdf.index.min(), pdf.index.max()
print(
    f"Built TrajectoryCollection with {len(traj_collection.trajectories)} trajectories "
    f"({N_SAMPLES} waypoints each)"
)
print(f"Animated time range: {t_min_idx}  ->  {t_max_idx}")
print(f"  Duration: {(t_max_idx - t_min_idx)}")
print(
    "To see motion, click the triangular Play button on the widget below the "
    "map. TripsLayer only draws data at its current_time -- without pressing "
    "Play, time never advances."
)

# Color per route (MTA-ish: 4 = green, 3 = red).
route_rgb = {"4": [0, 147, 60], "3": [238, 53, 46]}
traj_colors = np.array(
    [
        route_rgb.get(traj.df["route_id"].iloc[0], [128, 128, 128])
        for traj in traj_collection.trajectories
    ],
    dtype=np.uint8,
)

trips_layer = TripsLayer.from_movingpandas(
    traj_collection,
    get_color=traj_colors,
    width_min_pixels=10,  # chunkier trains so they're easy to spot
    trail_length=360,  # seconds of fading trail behind each train
    cap_rounded=True,
    joint_rounded=True,
    opacity=0.95,
    fade_trail=True,
)

# Thin static underlay of the route lines, so you can see the corridor even
# between trips.
static_routes = (
    df_routes.filter(pl.col("id").is_in([t["route_id"] for t in anim_trips]))
    .rename({"geometry": "wkb", "id": "route_id"})
    .with_columns(geom=st.from_wkb("wkb"))
    .select(["route_id", "geom"])
    .to_pandas()
)
static_gdf = gpd.GeoDataFrame(
    {"route_id": static_routes["route_id"]},
    geometry=gpd.GeoSeries.from_wkb(static_routes["geom"]),
    crs="EPSG:4326",
)
underlay = PathLayer.from_geopandas(
    static_gdf,
    get_color=[160, 160, 160, 180],
    width_min_pixels=3,
)

m = Map([underlay, trips_layer], height=650)

# Advance 10 "data seconds" per animation frame at 30 fps, i.e. 300x real time.
controller = trips_layer.animate(
    step=timedelta(seconds=10), fps=30, strftime_fmt="%H:%M:%S"
)

# Bundle map + controller in a VBox so the Play button is directly below the map.
widgets.VBox([m, controller])